# 🚀 AI Agents Workshop using Gemini API

This notebook covers:
- Prompt Engineering
- AI Agents
- Tool usage
- Retrieval-Augmented Generation (RAG)
- Vector Database + Indexing

👉 Only Gemini API is used.

In [ ]:
!pip install -q google-generativeai faiss-cpu numpy pandas

## 📦 Import Libraries

In [ ]:
import google.generativeai as genai
import numpy as np
import pandas as pd
import faiss

## 🔑 Configure Gemini API

In [ ]:
API_KEY = "YOUR_GEMINI_API_KEY"

genai.configure(api_key=API_KEY)
model = genai.GenerativeModel("gemini-1.5-flash")

## ✏️ Basic Prompt

In [ ]:
response = model.generate_content("Explain AI in 2 lines")
print(response.text)

## 🎭 Prompt Engineering - Role Prompting

In [ ]:
prompt = """
You are a senior AI engineer.
Explain neural networks simply.
"""

print(model.generate_content(prompt).text)

## 🧠 Few-shot Prompting

In [ ]:
prompt = """
Input: Apple
Output: Fruit

Input: Carrot
Output: Vegetable

Input: Chicken
Output:
"""

print(model.generate_content(prompt).text)

## 🤖 Simple AI Agent

In [ ]:
def simple_agent(query):
    prompt = f"You are an AI agent. Answer clearly: {query}"
    return model.generate_content(prompt).text

print(simple_agent('What is reinforcement learning?'))

## 🛠 Tool-based Agent

In [ ]:
def calculator_tool(expr):
    return eval(expr)

def agent_with_tool(query):
    if 'calculate' in query:
        expr = query.split('calculate')[-1].strip()
        return calculator_tool(expr)
    return model.generate_content(query).text

print(agent_with_tool('calculate 10*5'))

## 📚 RAG Setup

In [ ]:
documents = [
    'Machine learning is a subset of AI.',
    'Deep learning uses neural networks.',
    'Reinforcement learning uses rewards.'
]

In [ ]:
def get_embedding(text):
    emb = genai.embed_content(
        model='models/embedding-001',
        content=text
    )
    return np.array(emb['embedding'])

embeddings = np.array([get_embedding(doc) for doc in documents])

In [ ]:
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

def retrieve(query, k=2):
    q_vec = np.array([get_embedding(query)])
    _, idx = index.search(q_vec, k)
    return [documents[i] for i in idx[0]]

def rag_pipeline(query):
    context = retrieve(query)
    prompt = f"Context: {context}\nQuestion: {query}"
    return model.generate_content(prompt).text

print(rag_pipeline('Explain deep learning'))